# Jev 소개와 Primitive별 사용 시나리오

Jev가 해결하려는 문제와 System One decision model의 설계 이유를 살펴보고, 세 가지 primitive인 **Choice**, **Noul**, **Score**를 각각 하나의 시나리오로 작성합니다.

> 이 노트북의 예제 셀은 실행하지 않은 상태입니다. 실제 실행에는 TypeSafe SDK 설치, API 접근 권한, `TYPESAFE_API_KEY` 설정이 필요합니다.


## 1. Jev의 목적

Jev의 목적은 **소프트웨어가 바로 사용할 수 있는 빠르고 구조화된 판단**을 만드는 것입니다.

일반적인 LLM은 자연어를 생성하는 데 최적화되어 있습니다. 반면 실제 자동화 코드에서는 긴 답변보다 다음과 같은 작은 판단이 자주 필요합니다.

- 고객 문의를 어느 팀으로 보낼 것인가?
- 이 거래가 위험하다는 명제가 참일 확률은 얼마인가?
- 장애 심각도가 정의된 단계 중 어디에 가까운가?

이런 판단을 일반 LLM에 요청하면 모델은 답을 문자열로 생성하고, 애플리케이션은 문자열을 다시 파싱하고 검증해야 합니다. Jev는 가능한 출력 형태를 미리 제한하고 **typed value와 확률**을 직접 반환하여 판단을 `if`, 라우팅, 정렬, 임계값 같은 코드와 결합하는 것을 목표로 합니다.

```text
unstructured / structured state
              ↓
       focused question
              ↓
            Jev
              ↓
typed probabilistic decision
              ↓
 application code / workflow
```


## 2. Jev란 무엇인가?

Jev는 TypeSafe AI가 공개한 첫 번째 **System One Model**입니다. 공식 설명은 Jev를 다음과 같은 인터페이스로 소개합니다.

> unstructured state in, typed probabilistic decisions out

핵심 특성은 다음과 같습니다.

1. **텍스트를 생성하지 않는다.** 미리 정의한 후보, 순서형 level 또는 참/거짓에 대해 값을 반환합니다.
2. **출력 타입이 정해져 있다.** 모델이 후보 목록 밖의 문자열을 만들어 내지 않습니다.
3. **확률 분포를 반환한다.** 가장 높은 항목뿐 아니라 후보별 불확실성을 코드에서 사용할 수 있습니다.
4. **여러 질문을 병렬로 평가한다.** 같은 state를 보는 독립적인 질문을 한 요청에 담을 수 있습니다.
5. **작은 판단을 코드로 합성한다.** 모델에게 전체 업무 절차를 맡기기보다, 모델의 focused judgment를 일반 코드가 조합합니다.

Jev를 단순히 채팅 LLM의 작은 버전으로 보기보다는, 언어로 표현된 state에 대해 빠른 확률적 결정을 내리는 별도 모델 계열로 이해하는 편이 정확합니다. 공개 자료는 새로운 architecture, parallel sampler, 그리고 RLCD(Reinforcement Learning for Calibrated Decisions)를 사용한다고 설명하지만 세부 구현은 공개하지 않았습니다.


## 3. 왜 System One decision model을 개발했는가?

System One이라는 이름은 빠르고 직관적인 판단을 뜻하는 심리학의 System 1 개념에서 가져왔습니다. 여기서는 사람의 인지 이론을 그대로 구현한다는 의미보다, **짧고 집중된 판단을 빠르게 반복하는 계산 인터페이스**에 가깝습니다.

| System 1 - 《생각에 관한 생각(Thinking, Fast and Slow)》에 등장하는 개념으로, 빠르고 자동적이며 직관적인 사고 모드

### 기존 생성형 LLM을 자동화에 넣을 때 생기는 문제

| 문제 | 자동화에서의 영향 | Jev의 접근 |
| --- | --- | --- |
| token을 순차적으로 생성 | 판단 하나에도 출력 latency와 비용 발생 | 가능한 결과를 병렬 평가 |
| 자유 형식 문자열 | 파싱 실패, schema 위반, 후보 밖 답변 가능 | Choice·Noul·Score의 typed output |
| 질문 하나에 복잡한 업무 전체를 위임 | 판단 근거와 제어 흐름이 prompt 안에 묻힘 | 작은 질문으로 분해하고 코드에서 합성 |
| confidence를 자연어로 요청 | 일관되지 않거나 과신할 수 있음 | 모든 판단을 확률 형태로 반환하도록 설계 |
| 같은 state에 여러 번 질의 | 공통 입력을 반복 처리 | 같은 state의 독립 질문을 한 요청으로 평가 |

### 적합한 문제

- 사람이 충분한 맥락을 보고 빠르게 내릴 수 있는 focused judgment
- 가능한 결과가 사전에 정의된 분류, 라우팅, 위험 판단, 수준 평가
- 확률에 따라 자동 처리, 추가 확인, 사람 검토를 나누는 workflow

### 덜 적합한 문제

- 긴 계획을 세워야 하는 문제
- 새로운 자연어·코드·문서를 생성해야 하는 문제
- 한 질문의 답을 사용해야 다음 질문을 만들 수 있는 순차 추론

복잡한 업무는 작은 독립 판단들로 분해한 뒤 애플리케이션 코드에서 합성합니다. 실제로 앞 질문의 답이 있어야 다음 질문을 구성할 수 있다면 두 번째 요청을 보냅니다.


## 4. 세 가지 Primitive

| Primitive | 질문 형태 | 주요 반환값 | 코드에서의 일반적인 사용 |
| --- | --- | --- | --- |
| `Choice` | 정해진 후보 중 무엇인가? | `choice`, `probabilities`, `confidence` | 분류, 라우팅, action 선택 |
| `Noul` | 명제가 참인가? | `noul` = `P(yes)` | 임계값, guardrail, 조건문 |
| `Score` | 순서가 있는 수준 중 어디인가? | `score`, `legend`, `probabilities`, `confidence` | 우선순위, 심각도, 품질 수준 |

질문 ID는 응답을 찾기 위한 transport key이며 모델에 전달되는 질문 내용이 아닙니다. 따라서 `instructions` 안에 판단 조건을 완전하게 작성해야 합니다.


In [ ]:
# API key 불러오기
from dotenv import load_dotenv
import os

load_dotenv()

# 시나리오 1 — Choice: 고객 문의 라우팅

## 상황

고객 지원 시스템이 문의 내용과 주문 상태를 읽고 담당 팀을 선택합니다. 선택지는 서로 순서가 없는 업무 부서입니다.

## 왜 Choice인가?

최종 코드 경로가 `billing`, `technical`, `account`, `other` 중 하나로 나뉘기 때문입니다. 후보별 확률을 이용하면 확실한 문의는 자동 배정하고, 분포가 평평한 문의는 triage queue로 보낼 수 있습니다.

## 설계 포인트

- option key와 설명을 함께 제공합니다.
- 후보 목록이 모든 상황을 포괄하지 못할 수 있으므로 `other`를 포함합니다.
- “주요 문제”를 고르도록 지시해 부가적인 사실에 끌리지 않게 합니다.


In [8]:
from typesafe_sdk import Choice, TypeSafeClient

state = {
    "ticket": {
        "subject": "Duplicate charge",
        "message": "Order A-104 was charged twice. Please refund the duplicate charge.",
    },
    "service_status": "The customer can use every product feature normally.",
}

questions = {
    "route_team": Choice(
        instructions=(
            "Which team should handle the customer's main request in `ticket.message`? "
            "Use `service_status` only to distinguish a payment issue from an outage."
        ),
        criteria={
            "billing": "Charges, invoices, payments, and refunds.",
            "technical": "Software defects, broken features, and service outages.",
            "account": "Sign-in, identity verification, and account access.",
            "other": "The request does not fit the other teams.",
        },
    )
}

with TypeSafeClient() as client:
    response = client.system_one(state=state, questions=questions)

answer = response.answers["route_team"]
print("## Answer")
print("selected:", answer.choice)
print("probabilities:", answer.probabilities)
print("confidence:", answer.confidence)

selected_probability = answer.probabilities[answer.choice]
workflow = "auto_route" if selected_probability >= 0.85 else "human_triage"
print("workflow:", workflow)

# 사용 Resources
print("\n## Resources")
print("Model:", response.model)
print("Tokens:", response.usage)


## Answer
selected: billing
probabilities: {'other': 0.0, 'billing': 1.0, 'account': 0.0, 'technical': 0.0}
confidence: 1.0
workflow: auto_route

## Resources
Model: jev-1.13.0
Tokens: input_tokens=444 output_tokens=46


# 시나리오 2 — Noul: 결제 이상 징후 확인

## 상황

결제 이벤트와 계정 활동을 보고 “이 거래를 즉시 보류할 만큼 명확한 이상 징후가 있는가?”라는 하나의 명제를 평가합니다.

## 왜 Noul인가?

필요한 값이 yes/no label보다 **yes일 확률**이기 때문입니다. 애플리케이션은 운영 비용과 위험 허용도에 따라 임계값을 바꿀 수 있습니다.

- 높은 확률: 거래 보류
- 중간 확률: 추가 인증
- 낮은 확률: 정상 처리

`0.5`는 위험 수준이 중간이라는 의미가 아니라 yes와 no에 비슷한 확률을 준다는 의미입니다.


In [ ]:
from typesafe_sdk import Noul, TypeSafeClient

state = {
    "transaction": {
        "amount_usd": 920,
        "country": "US",
        "device_id": "new-device-91",
        "attempts_last_10_minutes": 4,
    },
    "account_history": {
        "usual_country": "KR",
        "usual_purchase_range_usd": [20, 120],
        "known_device": False,
    },
}

questions = {
    "clear_fraud_signal": Noul(
        instructions=(
            "Is there clear evidence in `transaction` and `account_history` that this "
            "transaction should be held for fraud review? Judge only the supplied facts."
        ),
        criteria={
            "true": "Multiple supplied signals strongly deviate from the account's normal behavior.",
            "false": "The supplied evidence is normal, weak, or insufficient to justify a fraud hold.",
        },
    )
}

with TypeSafeClient() as client:
    response = client.system_one(state=state, questions=questions)

p_hold = response.answers["clear_fraud_signal"].noul
print("## Answer")
print("P(hold for fraud review):", p_hold)

if p_hold >= 0.85:
    action = "hold_transaction"
elif p_hold >= 0.45:
    action = "request_step_up_authentication"
else:
    action = "allow_transaction"
print("action:", action)


# 사용 Resources
print("\n## Resources")
print("Model:", response.model)
print("Tokens:", response.usage)


# 시나리오 3 — Score: 서비스 장애 심각도

## 상황

관측된 장애 상태를 운영팀이 정의한 네 단계의 심각도 rubric에 배치합니다.

## 왜 Score인가?

결과에 자연스러운 순서가 있고, 인접 level 사이의 불확실성도 의미가 있기 때문입니다. `score`는 가장 높은 level 번호만 반환하는 것이 아니라 level index의 확률 가중 평균이므로 `1.7`처럼 두 단계 사이 값이 될 수 있습니다.

각 level 설명은 독립적으로 이해할 수 있게 작성해야 합니다. 배열 위치가 숫자 level이 됩니다.


In [ ]:
from typesafe_sdk import Score, TypeSafeClient

state = {
    "incident": {
        "affected_component": "checkout-api",
        "error_rate": 0.37,
        "affected_regions": ["ap-northeast-2"],
        "workaround": "Retry succeeds for about half of failed requests.",
        "data_loss": False,
    },
    "business_context": {
        "traffic_period": "peak",
        "customer_impact": "Some customers cannot complete payment on the first attempt.",
    },
}

questions = {
    "incident_severity": Score(
        instructions=(
            "Rate the current customer-facing severity using only `incident` and "
            "`business_context`. Do not infer unreported data loss or regions."
        ),
        criteria=[
            "No customer-visible impact; monitoring signal only.",
            "Limited customer impact with a reliable workaround.",
            "Major customer impact or unreliable workaround, but service is partly available.",
            "Widespread outage, critical safety issue, or confirmed data loss.",
        ],
    )
}

with TypeSafeClient() as client:
    response = client.system_one(state=state, questions=questions)

answer = response.answers["incident_severity"]
print("## Answer")
print("expected severity level:", answer.score)
print("level legend:", answer.legend)
print("probabilities:", answer.probabilities)
print("confidence:", answer.confidence)

if answer.score >= 2.5:
    workflow = "page_incident_commander"
elif answer.score >= 1.5:
    workflow = "open_high_priority_incident"
else:
    workflow = "standard_monitoring"
print("workflow:", workflow)

# 사용 Resources
print("\n## Resources")
print("Model:", response.model)
print("Tokens:", response.usage)


## 5. 세 시나리오에서 기억할 점

1. **state에는 판단에 필요한 증거를 넣습니다.** 모델이 알 수 없는 정보를 추측하게 만들지 않습니다.
2. **한 질문에는 하나의 focused judgment를 담습니다.** 분석·계획·실행 전체를 한 질문에 넣지 않습니다.
3. **criteria를 코드의 실제 분기와 맞춥니다.** 반환값을 다시 자연어로 해석하지 않아도 되게 설계합니다.
4. **확률로 운영 정책을 만듭니다.** 자동 처리, 추가 검사, 사람 검토의 임계값은 애플리케이션이 결정합니다.
5. **확률 합계가 1이라는 사실과 calibration은 다릅니다.** 실제 운영 데이터에서 confidence 구간별 정확도와 비용을 검증해야 합니다.
6. **같은 state의 독립 질문은 한 요청에 묶을 수 있습니다.** 한 답에 의존하는 다음 질문은 별도 요청으로 구성합니다.


# Jev의 backend와 tokenizer를 추정할 수 있을까?

## 현재 공개된 범위

TypeSafe의 공개 자료는 Jev에 대해 **새로운 model architecture**, **parallel sampler**, **RLCD**를 사용한다고 설명합니다. 그러나 다음 세부 사항은 공개 자료에서 확인되지 않습니다.

- 기반 backbone의 모델명과 parameter 수
- vocabulary와 tokenizer 종류
- `state`, `instructions`, `criteria`의 정확한 내부 직렬화 형식
- special token, system prefix, truncation 규칙
- token ID가 model 내부의 어떤 representation으로 바뀌는지

따라서 Jev가 특정 OpenAI 모델, Qwen 또는 다른 LLM을 backbone으로 쓴다고 단정할 수 없습니다.

## 실제 공개 입력 계약

요청 단위는 단순히 `state + question 문자열 하나`라기보다 다음 구조에 가깝습니다.

```text
state
└── questions
    ├── question id        # 응답을 연결하는 transport key; 모델 의미 입력에는 포함되지 않음
    └── question
        ├── type           # choice / noul / score
        ├── instructions   # 완전한 판단 질문
        └── criteria       # 후보, ordered levels, 또는 true/false 정의
```

같은 state에 여러 question을 보낼 수 있으며 각 질문은 독립적으로 평가됩니다. Choice와 Score에서는 criteria도 의미 입력이므로 token 비용을 생각할 때 함께 포함해야 합니다.


## `tiktoken`으로 알 수 있는 것과 없는 것

`tiktoken`은 OpenAI 모델에 사용되는 공개 encoding으로 **우리가 만든 plain text**를 token ID로 바꿉니다.

가능한 것:

- 동일한 가상 입력이 `o200k_base`, `cl100k_base` 등에서 몇 token인지 비교
- 한국어, JSON, option 설명이 token 수에 미치는 영향 확인
- OpenAI LLM baseline의 plain-text token 수를 근사

불가능한 것:

- Jev가 실제로 어떤 tokenizer를 사용하는지 식별
- Jev 내부 직렬화와 special token을 복원
- token 수만으로 backbone을 특정
- 우연히 token 수가 같다는 이유로 같은 tokenizer라고 결론 내리기

아래 코드는 **Jev 내부 입력의 복원 코드가 아니라**, 공개 필드를 우리가 명시적으로 문자열로 만든 뒤 OpenAI encoding별 차이를 관찰하는 실험입니다.

In [9]:
import json

import tiktoken

# 공개 API 필드를 이용해 만든 분석용 요청 예시
request = {
    "state": {
        "ticket": {
            "subject": "Duplicate charge",
            "message": "Order A-104 was charged twice. Please refund the duplicate charge.",
        },
        "service_status": "The customer can use every product feature normally.",
    },
    "questions": {
        "route_team": {
            "type": "choice",
            "instructions": "Which team should handle the customer's main request?",
            "criteria": {
                "billing": "Charges, invoices, payments, and refunds.",
                "technical": "Software defects, broken features, and service outages.",
                "account": "Sign-in, identity verification, and account access.",
                "other": "The request does not fit the other teams.",
            },
        }
    },
}


def analysis_render(payload: dict) -> str:
    """분석을 위해 우리가 정한 직렬화다. Jev의 내부 직렬화가 아니다."""
    parts = [
        "State:\n" + json.dumps(payload["state"], ensure_ascii=False, sort_keys=True),
    ]
    for question_id, question in payload["questions"].items():
        # question_id는 실제 모델 입력에서 제외된다는 공개 계약을 반영한다.
        parts.extend([
            "Question type: " + question["type"],
            "Instructions:\n" + question["instructions"],
            "Criteria:\n" + json.dumps(question.get("criteria"), ensure_ascii=False, sort_keys=True),
        ])
    return "\n\n".join(parts)


visible_text = analysis_render(request)
print(visible_text)


State:
{"service_status": "The customer can use every product feature normally.", "ticket": {"message": "Order A-104 was charged twice. Please refund the duplicate charge.", "subject": "Duplicate charge"}}

Question type: choice

Instructions:
Which team should handle the customer's main request?

Criteria:
{"account": "Sign-in, identity verification, and account access.", "billing": "Charges, invoices, payments, and refunds.", "other": "The request does not fit the other teams.", "technical": "Software defects, broken features, and service outages."}


In [12]:
def inspect_encoding(text: str, encoding_name: str, preview: int = 5) -> dict:
    encoding = tiktoken.get_encoding(encoding_name)
    token_ids = encoding.encode(text)
    # decode_single_token_bytes를 사용하면 공백과 UTF-8 조각을 잃지 않고 관찰할 수 있다.
    pieces = [encoding.decode_single_token_bytes(token).decode("utf-8", errors="replace")
              for token in token_ids[:preview]]
    return {
        "encoding": encoding_name,
        "token_count": len(token_ids),
        "first_token_ids": token_ids[:preview],
        "first_pieces": pieces,
        "round_trip_ok": encoding.decode(token_ids) == text,
    }


# 각 행은 '이 encoding을 가정했을 때'의 결과일 뿐이다.
encoding_names = ["o200k_base", "cl100k_base", "p50k_base"]
reports = [inspect_encoding(visible_text, name) for name in encoding_names]
for en, report in zip(encoding_names, reports):
    print(f"\n## Encoding report for {en}")
    print(json.dumps(report, ensure_ascii=False, indent=2))



## Encoding report for o200k_base
{
  "encoding": "o200k_base",
  "token_count": 118,
  "first_token_ids": [
    1881,
    734,
    10848,
    4912,
    10369
  ],
  "first_pieces": [
    "State",
    ":\n",
    "{\"",
    "service",
    "_status"
  ],
  "round_trip_ok": true
}

## Encoding report for cl100k_base
{
  "encoding": "cl100k_base",
  "token_count": 121,
  "first_token_ids": [
    1423,
    512,
    5018,
    8095,
    4878
  ],
  "first_pieces": [
    "State",
    ":\n",
    "{\"",
    "service",
    "_status"
  ],
  "round_trip_ok": true
}

## Encoding report for p50k_base
{
  "encoding": "p50k_base",
  "token_count": 137,
  "first_token_ids": [
    9012,
    25,
    198,
    4895,
    15271
  ],
  "first_pieces": [
    "State",
    ":",
    "\n",
    "{\"",
    "service"
  ],
  "round_trip_ok": true
}


## Jev를 실제로 호출하고 input token 자동 추출

아래 셀은 위에서 만든 `request`와 같은 내용을 TypeSafe SDK의 `Choice`로 변환해 Jev에 보냅니다. 응답이 객체 또는 딕셔너리 중 어느 형태로 오더라도 `usage.input_tokens`에 해당하는 값을 찾아 `jev_input_tokens`로 저장합니다. 이 값이 Jev 서비스가 보고한 실제 입력 token 사용량입니다.

> 이 셀은 API 요청을 발생시킵니다. `TYPESAFE_API_KEY`가 설정된 환경에서 직접 실행하세요.


In [14]:
from collections.abc import Mapping

from typesafe_sdk import Choice, TypeSafeClient


def find_token_count(value, aliases: tuple[str, ...], seen: set[int] | None = None) -> int | None:
    """SDK 객체나 dict 안에서 token count 필드를 재귀적으로 찾는다."""
    if value is None or isinstance(value, (str, bytes, int, float, bool)):
        return None

    seen = seen or set()
    object_id = id(value)
    if object_id in seen:
        return None
    seen.add(object_id)

    if hasattr(value, "model_dump"):
        value = value.model_dump()
    elif hasattr(value, "dict") and callable(value.dict):
        value = value.dict()

    if isinstance(value, Mapping):
        for key in aliases:
            if key in value and value[key] is not None:
                return int(value[key])

        # 보통 usage 아래에 있지만 SDK 버전별 wrapper 차이도 처리한다.
        priority_keys = ("usage", "token_usage", "meta", "metadata")
        nested_values = [value[key] for key in priority_keys if key in value]
        nested_values += [item for key, item in value.items() if key not in priority_keys]
        for nested in nested_values:
            found = find_token_count(nested, aliases, seen)
            if found is not None:
                return found
        return None

    for key in aliases:
        if hasattr(value, key):
            count = getattr(value, key)
            if count is not None:
                return int(count)

    for key in ("usage", "token_usage", "meta", "metadata"):
        if hasattr(value, key):
            found = find_token_count(getattr(value, key), aliases, seen)
            if found is not None:
                return found
    return None


question_payload = request["questions"]["route_team"]
jev_questions = {
    "route_team": Choice(
        instructions=question_payload["instructions"],
        criteria=question_payload["criteria"],
    )
}

with TypeSafeClient() as client:
    tiktoken_example_response = client.system_one(
        state=request["state"],
        questions=jev_questions,
    )

jev_input_tokens = find_token_count(
    tiktoken_example_response,
    aliases=("input_tokens", "inputTokens", "input_token_count"),
)
jev_output_tokens = find_token_count(
    tiktoken_example_response,
    aliases=("output_tokens", "outputTokens", "output_token_count"),
)

if jev_input_tokens is None:
    raise KeyError("Jev 응답에서 input token 사용량을 찾지 못했습니다.")

print("model:", tiktoken_example_response.model)
print("answer:", tiktoken_example_response.answers["route_team"].choice)
print("INPUT TOKENS:", jev_input_tokens)
print("OUTPUT TOKENS:", jev_output_tokens)


model: jev-1.13.0
answer: billing
INPUT TOKENS: 424
OUTPUT TOKENS: 46


## Jev의 실제 input token과 tiktoken 후보 비교

`jev_input_tokens`는 Jev 응답에서 자동 추출한 값입니다. 아래 결과는 우리가 만든 `visible_text`를 각 OpenAI encoding으로 세었을 때 얼마나 차이가 나는지 보여줍니다. 정확히 일치해도 Jev의 tokenizer가 확인된 것은 아닙니다. Jev 내부의 prefix, schema 구성, 직렬화 방식은 공개된 token 수만으로 알 수 없기 때문입니다.


In [15]:
def compare_with_jev_count(text: str, jev_count: int) -> list[dict]:
    rows = []
    for name in ["o200k_base", "cl100k_base", "p50k_base", "r50k_base"]:
        local_count = len(tiktoken.get_encoding(name).encode(text))
        rows.append({
            "encoding": name,
            "tiktoken_count": local_count,
            "jev_input_tokens": jev_count,
            "difference": local_count - jev_count,
            "exact_count_match": local_count == jev_count,
        })
    return sorted(rows, key=lambda row: abs(row["difference"]))


comparison = compare_with_jev_count(visible_text, jev_input_tokens)
for row in comparison:
    print(row)


{'encoding': 'p50k_base', 'tiktoken_count': 137, 'jev_input_tokens': 424, 'difference': -287, 'exact_count_match': False}
{'encoding': 'r50k_base', 'tiktoken_count': 137, 'jev_input_tokens': 424, 'difference': -287, 'exact_count_match': False}
{'encoding': 'cl100k_base', 'tiktoken_count': 121, 'jev_input_tokens': 424, 'difference': -303, 'exact_count_match': False}
{'encoding': 'o200k_base', 'tiktoken_count': 118, 'jev_input_tokens': 424, 'difference': -306, 'exact_count_match': False}


## 더 강한 증거를 얻으려면

같은 요청에서 문자열을 한 요소씩 바꾸며 서비스가 공개한 input token count를 기록합니다.

1. state에 ASCII 한 글자씩 추가
2. 한국어·emoji·긴 숫자·공백을 각각 추가
3. Choice option 하나를 추가하거나 설명만 변경
4. question ID만 변경
5. 같은 의미의 JSON key 순서만 변경

`question ID 변경 시 count 불변` 같은 성질은 입력 계약을 확인하는 데 유용합니다. 하지만 tokenizer를 특정하려면 token ID나 vocabulary/merge table처럼 더 직접적인 정보가 필요합니다. 현재 공개 API의 확률 출력만으로는 그 정보를 얻을 수 없습니다.


## 참고 자료

- [Introducing System One Models & Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [TypeSafe primitives](https://docs.typesafe.ai/primitives)
- [Choice](https://docs.typesafe.ai/primitives/choice)
- [Noul](https://docs.typesafe.ai/primitives/noul)
- [Score](https://docs.typesafe.ai/primitives/score)

공식 블로그의 속도·비용·성능 수치는 TypeSafe가 공개한 자체 측정 결과입니다. 독립적인 검증 결과와 구분해서 읽어야 합니다.
